# 02. Stage 1A: Content-Based Retrieval

Notebook này xây dựng tầng lọc thô dựa trên nội dung (Content-Based) để đề xuất các ứng viên ban đầu cho người dùng.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn TF-IDF + Cosine Similarity?**
    *   Dữ liệu phim từ TMDB chứa các đặc trưng từ khóa có cấu trúc rõ ràng: `genres`, `director`, `cast`, và `keywords`. Các đặc trưng này mang tính phân loại từ khóa rất cao. TF-IDF kết hợp biểu diễn n-gram là giải thuật cực kỳ tối ưu để định lượng mức độ quan trọng của từ khóa mà không lo bị quá khớp (overfitting). Cosine Similarity giúp tính toán độ tương đồng góc giữa các vector tần suất một cách nhanh chóng.
*   **Tại sao không chọn Deep Learning (Sentence-BERT)?**
    *   Sentence-BERT (SBERT) là mạng Transformer dùng để hiểu **ngữ nghĩa tự nhiên** của các câu văn tự do (như `overview`). Với dữ liệu từ khóa rời rạc (như genres hay tên diễn viên), SBERT không mang lại lợi ích về ngữ nghĩa mà còn gây ra Overhead tính toán cực lớn (tải model ~400MB, suy luận chậm trên CPU). TF-IDF là đủ và hiệu quả hơn rất nhiều cho keyword matching.
*   **Tại sao không chọn Word2Vec / FastText?**
    *   Các mô hình Word Embedding tĩnh yêu cầu khối lượng văn bản cực lớn để huấn luyện các mối quan hệ từ vựng, hoặc nếu dùng pre-trained thì thường không tối ưu cho các danh từ riêng (tên đạo diễn, diễn viên) hay thuật ngữ điện ảnh đặc thù.


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load phim
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))

# Xử lý missing values
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')
movies_df['keywords'] = movies_df['keywords'].fillna('')

# 1. Kết hợp đặc trưng dạng văn bản
def build_metadata_soup(row):
    genres = row['genres'].replace('|', ' ')
    cast = ' '.join(row['cast'].split('|')[:5])
    keywords = row['keywords'].replace('|', ' ')
    director = row['director'].replace(' ', '')
    return f"{genres} {director} {cast} {keywords}"

movies_df['soup'] = movies_df.apply(build_metadata_soup, axis=1)
display(movies_df[['title', 'soup']].head(3))


,title,soup
0,Avatar: Fire and Ash,Science Fiction Adventure Fantasy JamesCameron...
1,Graphic Desires,Thriller Crime AndyEdwards David Wayman Sian A...
2,Toy Story 4,Family Comedy Animation Adventure JoshCooley T...


In [2]:
# 2. Xây dựng TF-IDF Matrix và lưu trữ Vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies_df['soup'])

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

# Lưu trữ ma trận tương đồng và vectorizer
os.makedirs("models", exist_ok=True)
with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
    
with open("models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)


TF-IDF Matrix shape: (10000, 5000)


In [3]:
# 3. Định nghĩa hàm gợi ý Content-Based cho một danh sách phim đã xem
def get_content_based_candidates(liked_movie_ids, top_n=100):
    liked_idx = movies_df[movies_df['movieId'].isin(liked_movie_ids)].index.tolist()
    if not liked_idx:
        return movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(top_n).tolist()
        
    sim_scores = cosine_similarity(tfidf_matrix[liked_idx], tfidf_matrix).mean(axis=0)
    sorted_idx = np.argsort(sim_scores)[::-1]
    
    liked_idx_set = set(liked_idx)
    candidate_indices = [idx for idx in sorted_idx if idx not in liked_idx_set]
    
    recommended_movie_ids = movies_df.iloc[candidate_indices]['movieId'].head(top_n).tolist()
    return recommended_movie_ids

# Test thử nghiệm gợi ý
test_likes = [1339713, 1084244]
candidates = get_content_based_candidates(test_likes, top_n=5)
print("Phim đã xem:", movies_df[movies_df['movieId'].isin(test_likes)]['title'].tolist())
print("Gợi ý Content-based:", movies_df[movies_df['movieId'].isin(candidates)]['title'].tolist())


Phim đã xem: []
Gợi ý Content-based: ['Avatar: Fire and Ash', 'Graphic Desires', 'Toy Story 4', 'Interstellar', 'Zootopia 2']
